## step1: loading everything

In [1]:
import pandas as pd
import numpy as np
import yfinance as yf
import quantstats as qs
import warnings
warnings.filterwarnings('ignore')

features = pd.read_csv("../data/features.csv")
features['filing_date'] = pd.to_datetime(features['filing_date'])
print(features[['ticker', 'filing_date', 'composite_signal']].to_string(index=False))

ticker filing_date  composite_signal
  AAPL  2025-07-31         -0.028428
  AAPL  2025-10-30         -0.634671
  AAPL  2026-01-29         -0.246660
  AMZN  2025-07-31          1.202853
  AMZN  2025-10-30          1.544172
  AMZN  2026-02-05         -0.264927
 GOOGL  2025-10-29          0.459914
 GOOGL  2026-02-04         -0.622486
  META  2025-07-30          0.783658
  META  2025-10-29          0.380878
  META  2026-01-28         -1.575917
  MSFT  2025-04-30         -0.142280
  MSFT  2025-07-30         -0.227171
  MSFT  2025-10-29         -0.147725
  MSFT  2026-01-28         -0.481209


## step2: downloading the prices

In [2]:
TICKERS = ["AAPL", "MSFT", "GOOGL", "META", "AMZN"]

prices = yf.download(TICKERS, start="2025-01-01", end="2026-04-01", auto_adjust=True)['Close']
prices.index = pd.to_datetime(prices.index).tz_localize(None)

print(f"prices loaded: {prices.shape}")
print(prices.tail(3))

[*********************100%***********************]  5 of 5 completed

prices loaded: (311, 5)
Ticker            AAPL        AMZN       GOOGL        META        MSFT
Date                                                                  
2026-03-27  248.800003  199.339996  274.339996  525.719971  356.769989
2026-03-30  246.630005  200.949997  273.500000  536.380005  358.959991
2026-03-31  253.789993  208.270004  287.559998  572.130005  370.170013


## step3: building the long-short portfolio

In [3]:
# for each quarter sorting tickers by composite signal. go long top 2 tickers, short bottom 2 tickers

# getting unique quarters
features['quarter'] = features['filing_date'].dt.to_period('Q')
quarters = features['quarter'].unique()

portfolio_returns = []

for q in sorted(quarters):
    q_data = features[features['quarter'] == q].copy()
    
    if len(q_data) < 4:
        continue
    
    q_data = q_data.sort_values('composite_signal', ascending=False)
    long_tickers = q_data.iloc[:2]['ticker'].tolist()
    short_tickers = q_data.iloc[-2:]['ticker'].tolist()
    
    # getting the start and end of the next quarter for holding period
    next_q_start = q_data['filing_date'].max() + pd.Timedelta(days=1)
    next_q_end = next_q_start + pd.Timedelta(days=90)
    
    # getting returns for holding period
    period_prices = prices[next_q_start:next_q_end]
    if period_prices.empty:
        continue
    
    period_returns = period_prices.pct_change().dropna()
    
    # long-short return = avg long return - avg short return
    long_ret = period_returns[long_tickers].mean(axis=1)
    short_ret = period_returns[short_tickers].mean(axis=1)
    ls_ret = long_ret - short_ret
    
    portfolio_returns.append(ls_ret)
    print(f"Q{q}: Long {long_tickers} | Short {short_tickers} | Days: {len(ls_ret)}")

portfolio = pd.concat(portfolio_returns).sort_index()
portfolio.name = "Long-Short Portfolio"
print(f"\ntotal trading days in backtest: {len(portfolio)}")

Q2025Q3: Long ['AMZN', 'META'] | Short ['AAPL', 'MSFT'] | Days: 63
Q2025Q4: Long ['AMZN', 'GOOGL'] | Short ['MSFT', 'AAPL'] | Days: 60
Q2026Q1: Long ['AAPL', 'AMZN'] | Short ['GOOGL', 'META'] | Days: 36

total trading days in backtest: 159


## step4: benchmark comparison with SPY

In [5]:
# downloading SPY as benchmark
spy = yf.download("SPY", start="2025-01-01", end="2026-04-01", auto_adjust=True)['Close']
spy.index = pd.to_datetime(spy.index).tz_localize(None)
if isinstance(spy, pd.DataFrame):
    spy = spy.squeeze()

spy_returns = spy.pct_change().dropna()
spy_returns.name = "SPY"

common_dates = portfolio.index.intersection(spy_returns.index)
port_aligned = portfolio[common_dates]
spy_aligned = spy_returns[common_dates]

print(f"Aligned trading days: {len(common_dates)}")

[*********************100%***********************]  1 of 1 completed

Aligned trading days: 159


## step5: performance metrics

In [6]:
# computing key metrics manually
def compute_metrics(returns, name):
    cumulative = (1 + returns).cumprod()
    total_return = cumulative.iloc[-1] - 1
    annualized = (1 + total_return) ** (252 / len(returns)) - 1
    volatility = returns.std() * np.sqrt(252)
    sharpe = annualized / volatility if volatility != 0 else np.nan
    max_dd = (cumulative / cumulative.cummax() - 1).min()
    
    print(f"{name}:")
    print(f"total return:{total_return:.2%}")
    print(f"annualized return:{annualized:.2%}")
    print(f"annualized vol:{volatility:.2%}")
    print(f"sharpe ratio:{sharpe:.2f}")
    print(f"max drawdown:{max_dd:.2%}")
    print()
    
    return {
        'strategy': name,
        'total_return': total_return,
        'annualized_return': annualized,
        'volatility': volatility,
        'sharpe': sharpe,
        'max_drawdown': max_dd
    }

port_metrics = compute_metrics(port_aligned, "Long-Short Portfolio")
spy_metrics = compute_metrics(spy_aligned, "SPY Benchmark")

Long-Short Portfolio:
total return:7.73%
annualized return:12.53%
annualized vol:23.08%
sharpe ratio:0.54
max drawdown:-20.42%

SPY Benchmark:
total return:5.67%
annualized return:9.13%
annualized vol:12.34%
sharpe ratio:0.74
max drawdown:-8.68%



### caveat:
- okay, so this method outperforms SPY in raw returns but after including risk-adjustment, SPY performs better.
- so, this means to make more money through this method, we gotta take more risk - positive alpha but poor risk management

## step6: saving data

In [7]:
# saving daily returns
perf_df = pd.DataFrame({
    'date': common_dates,
    'portfolio_return': port_aligned.values,
    'spy_return': spy_aligned.values
})
perf_df['portfolio_cumulative'] = (1 + perf_df['portfolio_return']).cumprod()
perf_df['spy_cumulative'] = (1 + perf_df['spy_return']).cumprod()

perf_df.to_csv("../data/portfolio_performance.csv", index=False)
print(perf_df.tail(5))

# saving metrics summary
metrics_df = pd.DataFrame([port_metrics, spy_metrics])
metrics_df.to_csv("../data/backtest_metrics.csv", index=False)

          date  portfolio_return  spy_return  portfolio_cumulative  \
154 2026-03-25          0.010227    0.005573              1.061525   
155 2026-03-26          0.047687   -0.017859              1.112145   
156 2026-03-27          0.003795   -0.017052              1.116366   
157 2026-03-30         -0.008930   -0.003343              1.106397   
158 2026-03-31         -0.026300    0.029068              1.077299   

     spy_cumulative  
154        1.067193  
155        1.048134  
156        1.030261  
157        1.026817  
158        1.056664  
